# SeineCrops — Sprint S4 : Divergence et phénologie

Deux volets complémentaires exploitant les séries temporelles
Sentinel-2 agrégées en S2 et le classifieur entraîné en S3.

**Divergence** : pour chaque parcelle, calcul d'un score de distance
au profil médian de sa classe déclarée (RPG). Les parcelles au-delà
d'un seuil sont signalées comme divergentes — erreur de déclaration,
culture intermédiaire non déclarée, ou stress exceptionnel.

**Phénologie** : ajustement d'une courbe lissée (Savitzky-Golay) sur
le profil NDVI temporel de chaque parcelle et extraction des métriques
SOS (Start of Season), POS (Peak of Season), EOS (End of Season) et
longueur de saison. Résultats agrégés par zone et par culture,
comparables aux produits HR-VPP Copernicus.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| 5.1 | Chargement et pivot — features wide depuis `derived.s2_parcelles_monthly`, labels depuis `derived.rpg_parcelles_aoi` |
| 5.2 | Profils médians par classe et scores de divergence (distance RMS, option DTW) |
| 5.3 | Lissage NDVI et extraction des métriques phénologiques (SOS, POS, EOS, LOS) |
| 5.4 | Chargement PostGIS et visualisation — tables `derived.divergence`, `derived.phenologie` |

---

## Dépendances

- `derived.s2_parcelles_monthly` — 11 458 381 lignes, 77 932 parcelles ×
  16 mois (sprint S2, `03_series_s2.ipynb` ; chiffre vérifié après le
  re-run nb03 — l'estimation initiale de 13,7 M lignes était datée)
- `derived.rpg_parcelles_aoi` — 80 689 lignes / 80 683 `id_parcel`
  distincts (6 doublons connus, cf. `methode.md` §3.2), colonnes
  `code_cultu` / `code_group` (sprint S1)
- `derived.s2_parcelles_ndvi_dates` — 2 595 821 lignes, 77 932 parcelles,
  166 dates distinctes, utilisée pour le lissage Savitzky-Golay en 5.3
  (69 534 parcelles avec `n_pixels >= 5`, seuil retenu en S3)

> ℹ️ nb03 a été rejoué sur les composites corrigés (fix nodata + fix CRS
> UTM 31N, cf. methode.md) et nb04 a validé un F1 macro 0.893 sur ce jeu
> propre. Les seuils et paramètres de ce notebook (percentile de
> divergence, seuil bimodal, fenêtre Savitzky-Golay) sont donc calibrés
> ici pour la première fois sur données correctes — les valeurs issues
> des essais précédents (notamment le seuil bimodal à 30 000, tracé au
> mois d'août 2024 défectueux, désormais corrigé) sont à revalider et ne
> doivent pas être reprises telles quelles.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkt as shapely_wkt
from scipy.linalg import solveh_banded
from scipy.sparse import diags
from dotenv import load_dotenv

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Connexion PostGIS ─────────────────────────────────────────────────
PG_PARAMS = {
    "host": os.getenv("PG_HOST", "localhost"),
    "port": int(os.getenv("PG_PORT", 5432)),
    "dbname": os.getenv("PG_DBNAME", "seinecrops"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}

# ── Constantes ────────────────────────────────────────────────────────
MONTHS = [f"{y}-{m:02d}" for y in (2023, 2024)
          for m in range(1, 13)
          if (y == 2023 and m >= 9) or (y == 2024)]  # 16 mois

# Mapping code_group → classe cible (v3 : 8 classes)
# Synchronise avec nb04 — betterave traitee via code_cultu == "BTN"
GROUP_MAP = {
    1:  "cereales_hiver",      # Ble tendre
    3:  "cereales_hiver",      # Orge
    2:  "mais",                # Mais grain et ensilage
    5:  "colza",               # Colza
    9:  "lin",                 # Plantes a fibres (≈ lin fibre en Normandie)
    18: "prairie",             # Prairies permanentes
    19: "prairie",             # Prairies temporaires
    25: "legumes_fleurs",      # Legumes ou fleurs
}


### 5.1 — Chargement et pivot

Chargement des features depuis `derived.s2_parcelles_monthly` (format
long) et des labels depuis `derived.rpg_parcelles_aoi`. Pivot en format
wide (une ligne par parcelle × 704 colonnes) et regroupement des cultures
via `GROUP_MAP`. Les parcelles hors dictionnaire sont classées `"autres"`.

In [ ]:
# ── Chargement depuis PostGIS ─────────────────────────────────────────
conn = psycopg2.connect(**PG_PARAMS)

df_long = pd.read_sql("""
    SELECT id_parcel, mois, variable, mean, std, p10, p90
    FROM derived.s2_parcelles_monthly
    ORDER BY id_parcel, mois, variable
""", conn)

df_labels = pd.read_sql("""
    SELECT id_parcel, code_group, code_cultu
    FROM derived.rpg_parcelles_aoi
""", conn)

conn.close()

# 6 id_parcel en doublon dans le RPG (cf. methode.md §3.2) — même fix que nb04
df_labels = df_labels.drop_duplicates(subset="id_parcel", keep="first")

print(f"Features long : {len(df_long):>12,} lignes")
print(f"Parcelles RPG : {len(df_labels):>12,} parcelles (dédupliquées)")
print(f"Parcelles S2  : {df_long['id_parcel'].nunique():>7,}")

In [ ]:
# ── Pivot long → wide ─────────────────────────────────────────────────
STATS = ["mean", "std", "p10", "p90"]

df_melt = df_long.melt(
    id_vars=["id_parcel", "mois", "variable"],
    value_vars=STATS,
    var_name="stat",
    value_name="value",
)
df_melt["col"] = df_melt["variable"] + "_" + df_melt["stat"] + "_" + df_melt["mois"]

df_wide = df_melt.pivot(
    index="id_parcel", columns="col", values="value"
).reset_index()
df_wide.columns.name = None
assert df_wide.shape[1] - 1 == 704, f"Attendu 704 features, obtenu {df_wide.shape[1] - 1}"

# ── Labels : regroupement des cultures ────────────────────────────────
df_labels["code_group"] = pd.to_numeric(df_labels["code_group"], errors="coerce")
df_labels["classe"] = df_labels["code_group"].map(GROUP_MAP).fillna("autres")
# Betterave : code_group 24 partagé avec d'autres cultures industrielles
df_labels.loc[df_labels["code_cultu"] == "BTN", "classe"] = "betterave"

df = df_wide.merge(df_labels[["id_parcel", "classe"]], on="id_parcel", how="inner")

print(f"Matrice wide  : {df.shape[0]:,} parcelles × {df.shape[1] - 2} features")
print(f"\nDistribution des classes :")
print(df["classe"].value_counts().to_string())

### 5.2 — Profils médians et scores de divergence

Les 704 features mélangent 7 bandes de réflectance brute (échelle DN,
~0-10 000) et 4 indices bornés (~-1 à 1) — cf. `methode.md` §3.1. Sans
standardisation, une distance serait dominée par les bandes brutes et
les indices n'y contribueraient quasiment pas. La **standardisation
(z-score par feature) est donc appliquée avant tout calcul de
distance**, pas en option a posteriori.

Pour chaque classe déclarée (RPG), calcul du profil médian sur les
704 features standardisées. La **distance RMS** (moyenne des carrés
d'écart sur les features valides, invariante au nombre de mois
observés) de chaque parcelle à son profil de classe mesure l'écart
entre le couvert observé par satellite et la culture déclarée. Un
score élevé signale une divergence : erreur de déclaration, culture
intermédiaire non déclarée, ou stress agronomique exceptionnel.

Les parcelles au-delà du seuil **médiane + 2 × IQR** intra-classe sont
signalées comme divergentes. Ce seuil est calculé ici pour la première
fois sur données corrigées (cf. note d'en-tête) et reste provisoire —
à valider par comparaison avec un échantillon de terrain ou d'imagerie
haute résolution.

In [ ]:
# ── Standardisation (obligatoire — mélange bandes brutes / indices) ───
# 7 bandes en réflectance brute (~0-10 000) + 4 indices bornés (~-1 à 1) :
# sans standardisation, la distance serait dominée par les bandes brutes.
# StandardScaler de sklearn refuse les NaN, donc standardisation manuelle
# via nanmean/nanstd (colonnes déjà ajoutées par une exécution précédente
# exclues, pour rester idempotent en cas de ré-exécution de la cellule).
NON_FEATURE_COLS = {"id_parcel", "classe", "dist_classe",
                     "pct_features_valides", "seuil_div", "divergent", "pop"}
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

X_raw = df[feature_cols].to_numpy(dtype=np.float64)
mu = np.nanmean(X_raw, axis=0)
sigma = np.nanstd(X_raw, axis=0)
sigma[sigma == 0] = 1.0  # features constantes → éviter division par zéro
X_scaled = (X_raw - mu) / sigma

df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled["classe"] = df["classe"].values

# ── Profil médian par classe déclarée (sur features standardisées) ────
medians = df_scaled.groupby("classe")[feature_cols].median()

print(f"Profils médians : {medians.shape[0]} classes × {medians.shape[1]} features")
print(f"NaN dans les profils médians : {medians.isna().sum().sum()}")

# ── Distance RMS au profil médian (nanmean, invariante à la couverture) ─
def rms_distance(X, X_ref):
    """Distance RMS par feature valide (nanmean, NaN ignorés)."""
    return np.sqrt(np.nanmean((X - X_ref) ** 2, axis=1))

classes = df["classe"].values
X_med = medians.loc[classes].to_numpy()

df["dist_classe"] = rms_distance(X_scaled, X_med)

# Couverture par parcelle (diagnostic — une distance basée sur trop peu
# de features valides est peu fiable)
n_valid = (~np.isnan(X_raw)).sum(axis=1)
df["pct_features_valides"] = 100 * n_valid / len(feature_cols)
print(f"\nCouverture features — min {df['pct_features_valides'].min():.1f} % / "
      f"médiane {df['pct_features_valides'].median():.1f} %")
print()
print(df["dist_classe"].describe().to_string())

In [ ]:
# ── Seuil de divergence : médiane + 2 × IQR (intra-classe) ───────────
def divergence_stats(g):
    q1, q2, q3 = g.quantile([0.25, 0.50, 0.75])
    seuil = q2 + 2 * (q3 - q1)
    n_div = (g > seuil).sum()
    return pd.Series({"median": q2, "IQR": q3 - q1, "seuil": seuil,
                       "n_div": n_div, "taux": n_div / len(g)})

stats_div = df.groupby("classe")["dist_classe"].apply(divergence_stats).unstack()

# ── Visualisation ─────────────────────────────────────────────────────
classes = stats_div.sort_values("taux", ascending=False).index
n_classes = len(classes)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.ravel()

for i, cls in enumerate(classes):
    ax = axes[i]
    vals = df.loc[df["classe"] == cls, "dist_classe"]
    seuil = stats_div.loc[cls, "seuil"]
    taux = stats_div.loc[cls, "taux"]

    ax.hist(vals, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(seuil, color="firebrick", ls="--", lw=1.5,
               label=f"med + 2·IQR = {seuil:,.1f}")
    ax.set_title(f"{cls}\n{taux:.1%} divergentes", fontsize=10)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlabel("distance RMS (par feature)")

# Masquer l'axe restant si < 8 classes
for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribution des distances RMS au profil médian de classe (RPG déclaré)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Appliquer le seuil ────────────────────────────────────────────────
df["seuil_div"] = df["classe"].map(stats_div["seuil"])
df["divergent"] = df["dist_classe"] > df["seuil_div"]

n_div = df["divergent"].sum()
print(f"\nParcelles divergentes : {n_div:,} / {len(df):,} "
      f"({100 * n_div / len(df):.1f} %)")
print(f"\nDétail par classe :")
print(stats_div[["median", "IQR", "seuil", "n_div", "taux"]]
        .sort_values("taux", ascending=False)
        .to_string(formatters={"taux": "{:.1%}".format,
                                "median": "{:.3f}".format,
                                "IQR": "{:.3f}".format,
                                "seuil": "{:.3f}".format,
                                "n_div": "{:,.0f}".format}))

In [ ]:
# ── Répartition spatiale des parcelles divergentes ─────────────────────
# Projection XY (Lambert-93, aspect égal) plutôt que sur un seul axe.
conn = psycopg2.connect(**PG_PARAMS)
df_geo = pd.read_sql("""
    SELECT id_parcel,
           ST_X(ST_Centroid(geom)) AS x_centroid,
           ST_Y(ST_Centroid(geom)) AS y_centroid
    FROM derived.rpg_parcelles_aoi
""", conn)
conn.close()

# 6 id_parcel en doublon dans le RPG (cf. methode.md §3.2) — dédupliqué
# avant merge pour éviter tout fan-out.
df_geo = df_geo.drop_duplicates(subset="id_parcel", keep="first")

df_diag = df[["id_parcel", "classe", "dist_classe", "divergent"]].merge(df_geo, on="id_parcel")

fig, ax = plt.subplots(figsize=(10, 9))
for is_div, color, label, z in [(False, "lightgray", "conforme", 1),
                                 (True, "firebrick", "divergente", 2)]:
    sub = df_diag[df_diag["divergent"] == is_div]
    ax.scatter(sub["x_centroid"], sub["y_centroid"], s=1.5, alpha=0.5,
               color=color, label=f"{label} ({len(sub):,})", zorder=z)

ax.set_xlabel("X (Lambert-93, m)")
ax.set_ylabel("Y (Lambert-93, m)")
ax.set_aspect("equal")
ax.legend(markerscale=12, loc="upper right")
ax.set_title("Répartition spatiale des parcelles divergentes\n(seuil médiane + 2·IQR, par classe déclarée)")
plt.tight_layout()
plt.show()

n_div = df_diag["divergent"].sum()
print(f"Divergentes : {n_div:,} / {len(df_diag):,} ({100 * n_div / len(df_diag):.1f} %)")


### Diagnostic — bande de divergence accrue au raccord orbital (30UYV)

Deux lignes quasi rectilignes de complétude réduite et de divergence
accrue traversent le centre de l'AOI. Investigation complète documentée
dans `methode.md` (Limites documentées). Résumé :

- Coïncide avec le raccord entre les orbites relatives **51 et 94**
  sur la tuile **30UYV** (confirmé par superposition des empreintes
  CDSE) — zone couverte par une seule orbite plutôt que deux.
- Divergence concentrée sur les *bords* de la bande (pic à 5,7-7,5 %
  dans les 100 premiers mètres), pas diffuse sur toute sa largeur —
  écarte l'hypothèse d'un simple déficit de complétude et l'hypothèse
  radiométrique (view angle/BRDF, qui varierait graduellement).
- Cause retenue : la trace orbitale dérive d'une acquisition à
  l'autre (~1,3 km d'étendue mesurée sur 10 dates, orbite 51),
  élargissant artificiellement la zone de transition observée.
- **Pas un bug de traitement** : limite physique d'acquisition,
  stable en position moyenne.

Un flag `zone_raccord_orbital` est calculé ci-dessous pour ne pas
confondre ce bruit d'échantillonnage avec un signal agronomique lors
de l'interprétation des parcelles divergentes en 5.4.

In [ ]:
# ── Flag zone_raccord_orbital (raccord 51/94, 30UYV) ───────────────────
# cf. methode.md — Limites documentées. Empreinte représentative par
# orbite (f_valid_aoi max) comme proxy de la position moyenne du bord.
def get_footprint_wkt(product_id: str) -> str:
    """Empreinte (WKT) d'une scène via l'API CDSE OData."""
    url = (f"https://catalogue.dataspace.copernicus.eu/odata/v1/"
           f"Products({product_id})?$select=Footprint")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    raw = r.json()["Footprint"]
    return raw.split(";", 1)[1].rstrip("'")

catalogue = pd.read_parquet(PROJECT_ROOT / "data/raw/s2/catalogue_dedup.parquet")
scenes_raccord = (
    catalogue[(catalogue["tile_id"] == "30UYV")
              & (catalogue["orbit_relative"].isin([51, 94]))]
    .sort_values("f_valid_aoi", ascending=False)
    .groupby("orbit_relative")
    .first()
    .reset_index()
)
scenes_raccord["footprint_wkt"] = scenes_raccord["product_id"].apply(get_footprint_wkt)
gdf_raccord = gpd.GeoDataFrame(
    scenes_raccord,
    geometry=[shapely_wkt.loads(w) for w in scenes_raccord["footprint_wkt"]],
    crs="EPSG:4326",
).to_crs("EPSG:2154")

# ── Distance de chaque parcelle au raccord ─────────────────────────────
conn = psycopg2.connect(**PG_PARAMS)
df_geo = pd.read_sql("""
    SELECT id_parcel,
           ST_X(ST_Centroid(geom)) AS x_centroid,
           ST_Y(ST_Centroid(geom)) AS y_centroid
    FROM derived.rpg_parcelles_aoi
""", conn)
conn.close()

# 6 id_parcel en doublon dans le RPG — dédupliqué avant merge pour éviter
# tout fan-out (cellule idempotente si réexécutée plusieurs fois).
df_geo = df_geo.drop_duplicates(subset="id_parcel", keep="first")

points = gpd.GeoSeries(
    gpd.points_from_xy(df_geo["x_centroid"], df_geo["y_centroid"]), crs="EPSG:2154"
)
boundaries = [gdf_raccord.loc[gdf_raccord["orbit_relative"] == o, "geometry"].iloc[0].boundary
              for o in (51, 94)]
df_geo["dist_raccord"] = np.column_stack(
    [points.distance(b) for b in boundaries]
).min(axis=1)

# ── Flag et vérification (idempotent) ───────────────────────────────────
SEUIL_RACCORD = 2000  # m — largeur de la zone de transition observée
df = df.drop(columns=["dist_raccord", "zone_raccord_orbital"], errors="ignore")
df = df.merge(df_geo[["id_parcel", "dist_raccord"]], on="id_parcel", how="left")
df["zone_raccord_orbital"] = df["dist_raccord"] < SEUIL_RACCORD

n_flag = df["zone_raccord_orbital"].sum()
n_flag_div = (df["zone_raccord_orbital"] & df["divergent"]).sum()
print(f"Lignes df : {len(df):,} (attendu : 77 932)")
print(f"Parcelles proches d'un raccord orbital (< {SEUIL_RACCORD} m) : "
      f"{n_flag:,} ({100*n_flag/len(df):.1f} %)")
print(f"  … dont divergentes : {n_flag_div:,} "
      f"({100*n_flag_div/n_flag:.1f} % de cette zone, "
      f"vs {100*df['divergent'].mean():.1f} % en moyenne AOI)")


### 5.3 — Lissage NDVI et extraction phénologique

Les acquisitions dans `derived.s2_parcelles_ndvi_dates` sont irrégulières
(166 dates sur 16 mois). Plutôt qu'interpoler puis lisser (biais
d'interpolation linéaire + double lissage implicite), on utilise un
**lisseur de Whittaker** (moindres carrés pénalisés, pondérés par
`n_pixels`) — la référence du domaine pour ce type de données
(TIMESAT, HR-VPP Copernicus), nativement adaptée aux séries irrégulières :
chaque observation pèse selon sa fiabilité (`n_pixels`), sans valeur
fabriquée entre deux dates observées.

SOS, POS et EOS sont extraits par seuils d'amplitude (20 % de
l'amplitude saisonnière min→max), comme prévu dans `methode.md`.

La recherche de SOS/POS/EOS est restreinte à une fenêtre calendaire par classe (`FENETRES_PHENOLOGIE`, cellule 4) : le début de la fenêtre d'observation (Sept N) précède le semis des cultures de printemps/été (maïs, betterave...) — sans cette contrainte, le maximum détecté correspond souvent à la culture précédente ou à un couvert intermédiaire, pas à la culture déclarée (diagnostiqué via un taux `pos_en_bord` élevé, notamment 91,5 % pour la betterave et 95,8 % pour le maïs avant fenêtrage).

In [ ]:
# ── 5.3 — Chargement des profils NDVI par date (valeur + n_pixels) ─────
conn = psycopg2.connect(**PG_PARAMS)
df_ndvi_long = pd.read_sql("""
    SELECT id_parcel, date, mean AS ndvi, n_pixels
    FROM derived.s2_parcelles_ndvi_dates
    WHERE n_pixels >= 1
""", conn)
conn.close()

# psycopg2 renvoie un DATE SQL en datetime.date (dtype object) —
# conversion explicite en datetime64 pour les opérations .dt et
# l'arithmétique de dates des cellules suivantes.
df_ndvi_long["date"] = pd.to_datetime(df_ndvi_long["date"])

df_ndvi_long = df_ndvi_long[df_ndvi_long["id_parcel"].isin(df["id_parcel"])]

print(f"Lignes NDVI       : {len(df_ndvi_long):,}")
print(f"Parcelles         : {df_ndvi_long['id_parcel'].nunique():,}")
print(f"Dates distinctes  : {df_ndvi_long['date'].nunique():,}")
print(f"dtype date        : {df_ndvi_long['date'].dtype}")
print(f"n_pixels          : min {df_ndvi_long['n_pixels'].min()}, "
      f"médiane {df_ndvi_long['n_pixels'].median():.0f}")


In [ ]:
# ── Grille régulière (5 j) — chaque observation reste sur sa vraie date,
# juste "aimantée" au point de grille le plus proche (pas de valeur
# fabriquée entre deux observations, contrairement à l'interpolation) ───
date_min = df_ndvi_long["date"].min()
date_max = df_ndvi_long["date"].max()
grille_dates = pd.date_range(date_min, date_max, freq="5D")
jours_grille = (grille_dates - date_min).days.values

df_ndvi_long["jour"] = (df_ndvi_long["date"] - date_min).dt.days
df_ndvi_long["idx_grille"] = np.searchsorted(jours_grille, df_ndvi_long["jour"])
df_ndvi_long["idx_grille"] = df_ndvi_long["idx_grille"].clip(0, len(jours_grille) - 1)

# ── Agrégation vectorisée en cas de collision (plusieurs dates dans le
# même bin de 5 j) : moyenne pondérée par n_pixels ─────────────────────
df_ndvi_long["poids_ndvi"] = df_ndvi_long["ndvi"] * df_ndvi_long["n_pixels"]

agg = df_ndvi_long.groupby(["id_parcel", "idx_grille"], sort=False, as_index=False).agg(
    poids_ndvi_sum=("poids_ndvi", "sum"),
    n_pixels=("n_pixels", "sum"),
)
agg["ndvi"] = agg["poids_ndvi_sum"] / agg["n_pixels"]
df_binned = agg[["id_parcel", "idx_grille", "ndvi", "n_pixels"]]

df_valeurs = df_binned.pivot(index="id_parcel", columns="idx_grille", values="ndvi")
df_poids = df_binned.pivot(index="id_parcel", columns="idx_grille", values="n_pixels")

# Réindexer sur la grille complète et le même ensemble de parcelles que df
df_valeurs = df_valeurs.reindex(index=df["id_parcel"], columns=range(len(jours_grille)))
df_poids = df_poids.reindex(index=df["id_parcel"], columns=range(len(jours_grille)))

X_valeurs = df_valeurs.to_numpy(dtype=np.float64)
X_poids = np.nan_to_num(df_poids.to_numpy(dtype=np.float64), nan=0.0)
X_valeurs = np.nan_to_num(X_valeurs, nan=0.0)  # sans effet là où poids = 0

print(f"Grille : {len(grille_dates)} points, pas 5 j, "
      f"{grille_dates.min().date()} → {grille_dates.max().date()}")
print(f"Collisions résolues par agrégation : "
      f"{len(df_ndvi_long) - len(df_binned):,} observations regroupées")


In [ ]:
# ── Lisseur de Whittaker — moindres carrés pénalisés, pondérés n_pixels ──
# min_z  sum_i w_i (y_i - z_i)^2 + lambda * sum (D2 z)^2
# Système bandé pentadiagonal, résolu par parcelle (structure de pénalité
# partagée, calculée une seule fois : ~0,025 ms/parcelle, ~2 s au total).
LAMBDA_WHITTAKER = 800.0  # calibré visuellement (comparaison à 4 niveaux sur
# 4 parcelles-types) : assez fort pour lisser le bruit d'observation, assez
# modéré pour préserver un signal phénologique réel propre à une culture
# (ex. creux de floraison du colza, avril-mai) — cf. discussion nb05.
EPS_RIDGE = 1e-6  # sécurité numérique (parcelles à très peu d'observations)
N_MIN_OBS = 2  # sous ce seuil, système mal posé (noyau de D2'D2) → NaN

def build_D2_penalty_bands(n, lam):
    D2 = diags([1, -2, 1], [0, 1, 2], shape=(n - 2, n), dtype=np.float64).toarray()
    P = lam * (D2.T @ D2)
    u = 2
    ab = np.zeros((u + 1, n))
    for i in range(n):
        for j in range(max(0, i - u), i + 1):
            ab[u + j - i, i] = P[j, i]
    ab[-1, :] += EPS_RIDGE
    return ab

ab_D2 = build_D2_penalty_bands(len(jours_grille), LAMBDA_WHITTAKER)

X_smooth = np.full_like(X_valeurs, np.nan)
for i in range(X_valeurs.shape[0]):
    w = X_poids[i]
    if (w > 0).sum() < N_MIN_OBS:
        continue
    ab = ab_D2.copy()
    ab[-1, :] += w
    X_smooth[i] = solveh_banded(ab, w * X_valeurs[i], lower=False)

n_non_calculable = np.isnan(X_smooth).all(axis=1).sum()
print(f"Parcelles non calculables (< {N_MIN_OBS} observations pondérées) : "
      f"{n_non_calculable:,}")
print(f"NaN dans X_smooth : {np.isnan(X_smooth).sum():,} / {X_smooth.size:,}")


In [ ]:
# ── Extraction SOS / POS / EOS / LOS par seuils d'amplitude, avec fenêtre
# de recherche par classe ───────────────────────────────────────────────
# Convention HR-VPP/TIMESAT : seuils à 20 % de l'amplitude saisonnière ;
# SOS sur la branche montante, EOS sur la branche descendante, POS = date
# du maximum. Minimum local de chaque côté du pic (pas un minimum global
# partagé) : le creux pré-saison (interculture) et le creux post-récolte
# (sol nu) sont deux phénomènes différents, à des niveaux différents —
# un minimum global biaisait le seuil de l'un via le creux de l'autre.
#
# La fenêtre de 16 mois (Sept N → Déc N+1) déborde volontairement avant
# la campagne pour les besoins de la classification (nb04) — mais pour
# les cultures de printemps/été, le début de fenêtre correspond encore à
# la culture précédente ou à un couvert intermédiaire (CIPAN), pas à la
# culture déclarée. On restreint donc la RECHERCHE du maximum (pas le
# lissage, qui reste calculé sur tout le signal disponible) à une fenêtre
# calendaire par classe. Calendrier Normandie approximatif — à affiner.
#
# NB : si le vrai creux pré-saison (ou post-récolte) tombe hors fenêtre,
# le minimum local observable devient le bord de fenêtre lui-même —
# repousser jmin/jmax ne suffit pas à éliminer ce cas dans le cas général
# (le vrai creux peut être arbitrairement précoce/tardif). On le détecte
# donc explicitement (sos_en_bord / eos_en_bord, test d'indice — pas un
# test de seuil, qui serait tautologique avec un minimum local) plutôt
# que de chasser une fenêtre parfaite. Diagnostiqué sur maïs/betterave :
# pic isolé de SOS exactement sur jmin, largement avant le calendrier de
# semis réel — corrigé par ce flag, pas par un nouvel élargissement.
FENETRES_PHENOLOGIE = {
    "cereales_hiver": (60, 400),    # semis mi-oct N → récolte fin été N+1
    "colza":          (5, 380),     # semis fin août N → récolte mi-sept N+1
    "mais":           (230, 470),   # semis avril-mai N+1 → récolte sept-oct N+1
    "betterave":      (170, 475),   # semis mars-avril N+1 → récolte oct-déc N+1
                                     # (~fin de grille — cf. limite documentée ci-dessous)
    "lin":            (150, 350),   # semis mars N+1 → récolte juillet N+1
    "legumes_fleurs": (150, 450),   # classe hétérogène, marge large
    # prairie / autres : pas de fenêtre — couvert pérenne ou classe
    # hétérogène ; pos_en_bord/sos_en_bord/eos_en_bord restent le signal.
}
FRAC_SOS = 0.20
FRAC_EOS = 0.20

def extraire_phenologie(y, jours, fenetre=None):
    y_recherche = y.copy()
    if fenetre is not None:
        jmin, jmax = fenetre
        hors_fenetre = (jours < jmin) | (jours > jmax)
        y_recherche = np.where(hors_fenetre, np.nan, y)

    if np.isnan(y_recherche).all():
        return np.nan, np.nan, np.nan, np.nan, False, False, False

    i_max = np.nanargmax(y_recherche)
    y_max = y_recherche[i_max]

    y_min_avant = np.nanmin(y_recherche[:i_max + 1])
    y_min_apres = np.nanmin(y_recherche[i_max:])
    amplitude_avant = y_max - y_min_avant
    amplitude_apres = y_max - y_min_apres

    if amplitude_avant <= 0 and amplitude_apres <= 0:
        return np.nan, np.nan, np.nan, np.nan, False, False, False

    seuil_sos = y_min_avant + FRAC_SOS * amplitude_avant if amplitude_avant > 0 else y_max
    seuil_eos = y_min_apres + FRAC_EOS * amplitude_apres if amplitude_apres > 0 else y_max

    fenetre_valide = ~np.isnan(y_recherche)
    idx_valides = np.where(fenetre_valide)[0]

    avant_pic = np.where(fenetre_valide[:i_max + 1] & (y_recherche[:i_max + 1] <= seuil_sos))[0]
    idx_sos = avant_pic[-1] if len(avant_pic) else idx_valides[0]
    sos_en_bord = idx_sos <= idx_valides[0] + 1  # SOS retombé sur le bord gauche

    apres_pic = np.where(fenetre_valide[i_max:] & (y_recherche[i_max:] <= seuil_eos))[0]
    idx_eos = i_max + apres_pic[0] if len(apres_pic) else idx_valides[-1]
    eos_en_bord = idx_eos >= idx_valides[-1] - 1  # EOS retombé sur le bord droit

    pos_en_bord = i_max <= idx_valides[0] + 1 or i_max >= idx_valides[-1] - 1

    sos, eos, pos = jours[idx_sos], jours[idx_eos], jours[i_max]
    return sos, pos, eos, eos - sos, pos_en_bord, sos_en_bord, eos_en_bord

resultats = []
for i in range(X_smooth.shape[0]):
    classe_i = df["classe"].iloc[i]
    fenetre = FENETRES_PHENOLOGIE.get(classe_i)  # None pour prairie/autres
    resultats.append(extraire_phenologie(X_smooth[i], jours_grille, fenetre))

df_pheno = pd.DataFrame(
    resultats,
    columns=["sos_jour", "pos_jour", "eos_jour", "los_jours",
             "pos_en_bord", "sos_en_bord", "eos_en_bord"],
    index=df["id_parcel"],
).reset_index()

for col, out in [("sos_jour", "sos_date"), ("pos_jour", "pos_date"), ("eos_jour", "eos_date")]:
    df_pheno[out] = df_pheno[col].apply(
        lambda j: date_min + pd.Timedelta(days=j) if pd.notna(j) else pd.NaT
    )

df_pheno = df_pheno.merge(df[["id_parcel", "classe"]], on="id_parcel")
df_pheno["fiable"] = ~(df_pheno["pos_en_bord"] | df_pheno["sos_en_bord"] | df_pheno["eos_en_bord"])

n_nan = df_pheno["sos_jour"].isna().sum()
print(f"Phénologie extraite : {len(df_pheno) - n_nan:,} / {len(df_pheno):,} parcelles "
      f"(attendu : 77 932)")
print()
print(df_pheno.groupby("classe")[["sos_date", "pos_date", "eos_date", "los_jours"]]
      .agg({"sos_date": "median", "pos_date": "median", "eos_date": "median", "los_jours": "median"})
      .to_string())
print()

print(df_pheno.groupby("classe")[["sos_en_bord", "eos_en_bord", "pos_en_bord", "fiable"]].mean()
      .to_string(formatters={c: "{:.1%}".format
                              for c in ["sos_en_bord", "eos_en_bord", "pos_en_bord", "fiable"]}))
print()
print(df_pheno[df_pheno["fiable"]].groupby("classe")["los_jours"].median().to_string())


In [ ]:
# ── Validation visuelle : profil brut, lissé, repères phénologiques ────
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
axes = axes.ravel()
rng = np.random.default_rng(42)

for i, cls in enumerate(sorted(df["classe"].unique())):
    ax = axes[i]
    candidats = df_pheno.loc[(df_pheno["classe"] == cls) & df_pheno["sos_jour"].notna(), "id_parcel"]
    if candidats.empty:
        ax.set_visible(False)
        continue
    id_exemple = rng.choice(candidats.values)
    idx = df.index[df["id_parcel"] == id_exemple][0]

    obs = df_ndvi_long[df_ndvi_long["id_parcel"] == id_exemple]
    ax.scatter(obs["jour"], obs["ndvi"], s=10, alpha=0.4, color="gray", label="NDVI brut")
    ax.plot(jours_grille, X_smooth[idx], color="darkgreen", lw=1.5, label="lissé (Whittaker)")

    row = df_pheno[df_pheno["id_parcel"] == id_exemple].iloc[0]
    for jour, label, color in [(row["sos_jour"], "SOS", "steelblue"),
                                (row["pos_jour"], "POS", "firebrick"),
                                (row["eos_jour"], "EOS", "darkorange")]:
        ax.axvline(jour, color=color, ls="--", lw=1, label=label)

    ax.set_title(f"{cls}", fontsize=9)
    ax.legend(fontsize=6, loc="lower center")

fig.suptitle("Profils NDVI, lissage et repères phénologiques — un exemple par classe", y=1.02)
plt.tight_layout()
plt.show()


### 5.4 — Chargement PostGIS et visualisation

Persistance des résultats de 5.2 (`derived.divergence`) et 5.3
(`derived.phenologie`) dans deux tables dédiées, avec `ON CONFLICT DO
UPDATE` — les valeurs peuvent changer d'un run à l'autre (recalibrage
de `λ`, ajustement de `FENETRES_PHENOLOGIE`), comme pour les prédictions
de nb04. Upsert vectorisé via `psycopg2.extras.execute_values`, plus
rapide que `pandas.to_sql` sur 77 932 lignes.

In [ ]:
# ── DDL : tables derived.divergence et derived.phenologie ──────────────
conn = psycopg2.connect(**PG_PARAMS)
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS derived.divergence (
            id_parcel             text PRIMARY KEY,
            classe_declaree       text NOT NULL,
            dist_classe           double precision,
            seuil_div             double precision,
            divergent             boolean,
            dist_raccord          double precision,
            zone_raccord_orbital  boolean,
            version_pipeline      text NOT NULL,
            date_calcul           timestamp NOT NULL DEFAULT now()
        );

        CREATE TABLE IF NOT EXISTS derived.phenologie (
            id_parcel         text PRIMARY KEY,
            classe_declaree   text NOT NULL,
            sos_date          date,
            pos_date          date,
            eos_date          date,
            los_jours         integer,
            sos_en_bord       boolean,
            eos_en_bord       boolean,
            pos_en_bord       boolean,
            fiable            boolean,
            lambda_whittaker  double precision,
            version_pipeline  text NOT NULL,
            date_calcul       timestamp NOT NULL DEFAULT now()
        );
    """)
conn.commit()
conn.close()
print("Tables derived.divergence et derived.phenologie prêtes.")


In [ ]:
# ── Upsert derived.divergence ───────────────────────────────────────────
from psycopg2.extras import execute_values

VERSION_PIPELINE = "S4-v1"

cols_div = ["id_parcel", "classe", "dist_classe", "seuil_div", "divergent",
            "dist_raccord", "zone_raccord_orbital"]
df_divergence = df[cols_div].copy()
df_divergence["version_pipeline"] = VERSION_PIPELINE
# NaN → None pour psycopg2 (dist_raccord peut être NaN si merge incomplet)
df_divergence = df_divergence.where(pd.notna(df_divergence), None)

rows_div = list(df_divergence.itertuples(index=False, name=None))

def to_native(v):
    """Convertit un scalaire numpy/pandas en type Python natif pour psycopg2.
    numpy.float64 hérite de float et passe déjà nativement, mais numpy.int64
    et numpy.bool_ ne sont pas adaptables tels quels — d'où l'échec sur
    los_jours/les colonnes booléennes après passage par .where(..., None)
    (qui force le dtype object, où les valeurs restent "boxées" en numpy).
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    if isinstance(v, np.bool_):
        return bool(v)
    return v

rows_div = [tuple(to_native(v) for v in row) for row in rows_div]

# Garde-fou : rows_div doit correspondre au schéma de derived.divergence,
# pas être resté sur un tableau d'une autre cellule (cf. bug rencontré :
# nom de variable partagé entre les deux cellules d'upsert).
N_COLS_DIV = len(cols_div) + 1  # + version_pipeline
assert all(len(r) == N_COLS_DIV for r in rows_div[:5]), (
    f"rows_div ne correspond pas à derived.divergence : "
    f"{len(rows_div[0])} valeurs pour {N_COLS_DIV} colonnes attendues."
)

sql_div = """
    INSERT INTO derived.divergence
        (id_parcel, classe_declaree, dist_classe, seuil_div, divergent,
         dist_raccord, zone_raccord_orbital, version_pipeline)
    VALUES %s
    ON CONFLICT (id_parcel) DO UPDATE SET
        classe_declaree = EXCLUDED.classe_declaree,
        dist_classe = EXCLUDED.dist_classe,
        seuil_div = EXCLUDED.seuil_div,
        divergent = EXCLUDED.divergent,
        dist_raccord = EXCLUDED.dist_raccord,
        zone_raccord_orbital = EXCLUDED.zone_raccord_orbital,
        version_pipeline = EXCLUDED.version_pipeline,
        date_calcul = now()
"""

conn = psycopg2.connect(**PG_PARAMS)
with conn.cursor() as cur:
    execute_values(cur, sql_div, rows_div, page_size=5000)
conn.commit()
conn.close()
print(f"derived.divergence : {len(rows_div):,} lignes upsertées.")


In [ ]:
# ── Upsert derived.phenologie ───────────────────────────────────────────
cols_pheno = ["id_parcel", "classe", "sos_date", "pos_date", "eos_date",
              "los_jours", "sos_en_bord", "eos_en_bord", "pos_en_bord", "fiable"]
df_phenologie = df_pheno[cols_pheno].copy()
df_phenologie["lambda_whittaker"] = LAMBDA_WHITTAKER
df_phenologie["version_pipeline"] = VERSION_PIPELINE

# Types adaptés à la DDL : dates -> date (pas Timestamp), los_jours -> Int64
for c in ["sos_date", "pos_date", "eos_date"]:
    df_phenologie[c] = df_phenologie[c].dt.date
df_phenologie["los_jours"] = df_phenologie["los_jours"].astype("Int64")
df_phenologie = df_phenologie.where(pd.notna(df_phenologie), None)

rows_pheno = list(df_phenologie.itertuples(index=False, name=None))
rows_pheno = [tuple(to_native(v) for v in row) for row in rows_pheno]

# Garde-fou : rows_pheno doit correspondre au schéma de derived.phenologie.
N_COLS_PHENO = len(cols_pheno) + 2  # + lambda_whittaker + version_pipeline
assert all(len(r) == N_COLS_PHENO for r in rows_pheno[:5]), (
    f"rows_pheno ne correspond pas à derived.phenologie : "
    f"{len(rows_pheno[0])} valeurs pour {N_COLS_PHENO} colonnes attendues."
)

sql_pheno = """
    INSERT INTO derived.phenologie
        (id_parcel, classe_declaree, sos_date, pos_date, eos_date, los_jours,
         sos_en_bord, eos_en_bord, pos_en_bord, fiable, lambda_whittaker,
         version_pipeline)
    VALUES %s
    ON CONFLICT (id_parcel) DO UPDATE SET
        classe_declaree = EXCLUDED.classe_declaree,
        sos_date = EXCLUDED.sos_date,
        pos_date = EXCLUDED.pos_date,
        eos_date = EXCLUDED.eos_date,
        los_jours = EXCLUDED.los_jours,
        sos_en_bord = EXCLUDED.sos_en_bord,
        eos_en_bord = EXCLUDED.eos_en_bord,
        pos_en_bord = EXCLUDED.pos_en_bord,
        fiable = EXCLUDED.fiable,
        lambda_whittaker = EXCLUDED.lambda_whittaker,
        version_pipeline = EXCLUDED.version_pipeline,
        date_calcul = now()
"""

conn = psycopg2.connect(**PG_PARAMS)
with conn.cursor() as cur:
    execute_values(cur, sql_pheno, rows_pheno, page_size=5000)
conn.commit()
conn.close()
print(f"derived.phenologie : {len(rows_pheno):,} lignes upsertées.")


In [ ]:
# ── Vérification post-chargement ────────────────────────────────────────
conn = psycopg2.connect(**PG_PARAMS)
n_div = pd.read_sql("SELECT count(*) AS n FROM derived.divergence", conn)["n"].iloc[0]
n_pheno = pd.read_sql("SELECT count(*) AS n FROM derived.phenologie", conn)["n"].iloc[0]
n_div_flag = pd.read_sql(
    "SELECT count(*) AS n FROM derived.divergence WHERE divergent", conn
)["n"].iloc[0]
n_pheno_fiable = pd.read_sql(
    "SELECT count(*) AS n FROM derived.phenologie WHERE fiable", conn
)["n"].iloc[0]
conn.close()

print(f"derived.divergence : {n_div:,} lignes (attendu : 77 932), "
      f"dont {n_div_flag:,} divergentes")
print(f"derived.phenologie : {n_pheno:,} lignes (attendu : 77 932), "
      f"dont {n_pheno_fiable:,} fiables")
assert n_div == 77932, "Écart sur derived.divergence — vérifier les doublons avant merge"
assert n_pheno == 77932, "Écart sur derived.phenologie — vérifier les doublons avant merge"


#### Visualisation de synthèse

Croisement divergence (5.2) × phénologie (5.3) : une parcelle divergente
a-t-elle aussi une durée de saison (LOS) atypique par rapport à sa
classe ? Ce n'est pas garanti — la divergence porte sur la forme globale
du profil (704 features), pas spécifiquement sur SOS/POS/EOS — mais une
corrélation, si elle existe, renforcerait la crédibilité des deux
signaux indépendamment obtenus.

In [ ]:
# ── Distance de divergence vs durée de saison (LOS), par classe ────────
df_synth = df[["id_parcel", "classe", "dist_classe", "divergent"]].merge(
    df_pheno[["id_parcel", "los_jours", "fiable"]], on="id_parcel"
)
df_synth = df_synth[df_synth["fiable"]]  # phénologie fiable uniquement

classes_avec_fenetre = [c for c in df["classe"].unique() if c in FENETRES_PHENOLOGIE]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.ravel()

for i, cls in enumerate(classes_avec_fenetre):
    ax = axes[i]
    sub = df_synth[df_synth["classe"] == cls]
    for is_div, color, label in [(False, "lightgray", "conforme"),
                                  (True, "firebrick", "divergente")]:
        pts = sub[sub["divergent"] == is_div]
        ax.scatter(pts["dist_classe"], pts["los_jours"], s=6, alpha=0.4,
                   color=color, label=f"{label} ({len(pts):,})")
    ax.set_title(cls, fontsize=10)
    ax.set_xlabel("distance RMS (standardisée)")
    ax.set_ylabel("LOS (jours)")
    ax.legend(fontsize=7)

fig.suptitle("Divergence (5.2) vs longueur de saison (5.3) — parcelles fiables uniquement", y=1.01)
plt.tight_layout()
plt.show()

print(df_synth.groupby("divergent")["los_jours"].describe()[["mean", "std", "50%"]].to_string())
